# Group construction + POI/group KG + hyperbolic (RotH) training — NYC

Runs the three new pipeline stages that sit **before** the LLaDA fine-tune:

| Stage | Script | Produces |
|---|---|---|
| 1 | `group/build_groups.py` | real co-presence groups, tie graph, **group next-POI examples** |
| 2 | `group/build_kg.py` | merged POI + group knowledge graph (triples + hierarchy pairs) |
| 3 | `group/train_roth.py` | `poi_hyperbolic_embs_NYC.npy` — a **drop-in replacement** for `EMB_FILE` |
| 4 | `build_poi_poi_triples.py` | `poi_poi_triples_NYC.pt` for §6b's curvature-aware alignment |

**CPU only — set the accelerator to `None`.** Stage 3 is ~15 s/epoch on 4 cores
(9,445 entities, 175,727 triples, d=64). No GPU, no HF token, no Internet needed.

## What to attach as Kaggle input

**1. Data** — `yosrkharrat/kushflq` (or `eyamhamdi03/poi-final`). Must contain
`train_NYC.csv`, `val_NYC.csv`, `test_NYC.csv`, `poi_metadata_NYC.csv`. If the OLD
`poi_hyperbolic_embs.npy` is present, cell 5 prints the before/after D1 comparison.

**2. Code** — upload `group/*.py` + `build_poi_poi_triples.py` as a private Kaggle Dataset
(any name). Or set `GIT_REPO` in cell 0 and clone instead (needs Internet ON).

**You do not paste any paths.** Cell 0 finds both by content: the data dir is whichever
attached folder holds the CSVs, the code dir is whichever holds `build_groups.py` +
`affinity.py`.

> Every code cell below re-imports what it needs and reloads the paths from
> `_paths.json`, so a kernel restart or running cells out of order will not break it.
> Cell 0 still has to run once per session to write that file.

## 0 · Locate inputs  *(run this once per session)*

In [ ]:
import os, sys, glob, json, shutil, subprocess

# ── Option B only: clone the code instead of attaching it as a dataset (needs Internet ON) ──
GIT_REPO = ""          # e.g. "https://github.com/<you>/<repo>.git"
DATASET  = "NYC"

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./work"
os.makedirs(WORK, exist_ok=True)

def _find(pred):
    for root in ("/kaggle/input", WORK, "."):
        if not os.path.isdir(root):
            continue
        for dirpath, _dirs, files in os.walk(root):
            if pred(set(files)):
                return dirpath
    return None

DATA_DIR = _find(lambda fs: {f"train_{DATASET}.csv", f"poi_metadata_{DATASET}.csv"} <= fs)
assert DATA_DIR, ("no attached dataset has train_NYC.csv + poi_metadata_NYC.csv — "
                  "attach yosrkharrat/kushflq or eyamhamdi03/poi-final")
print("DATA_DIR :", DATA_DIR)
for f in sorted(os.listdir(DATA_DIR)):
    print("    ", f)

CODE_DIR = _find(lambda fs: {"build_groups.py", "affinity.py"} <= fs)
if CODE_DIR is None and GIT_REPO:
    subprocess.run(["git", "clone", "--depth", "1", GIT_REPO, f"{WORK}/code"], check=True)
    CODE_DIR = _find(lambda fs: {"build_groups.py", "affinity.py"} <= fs)
assert CODE_DIR, ("could not find the scripts — upload group/*.py + build_poi_poi_triples.py "
                  "as a Kaggle Dataset and attach it, or set GIT_REPO above")
print("\nCODE_DIR :", CODE_DIR)

# /kaggle/input is read-only and the scripts import each other, so copy somewhere writable.
# copyfile + chmod, NOT shutil.copy: copy() preserves the read-only bits from /kaggle/input,
# which makes re-running this cell fail with PermissionError.
RUN = f"{WORK}/code_run"
os.makedirs(RUN, exist_ok=True)
srcs = glob.glob(f"{CODE_DIR}/*.py")
_extra = os.path.join(os.path.dirname(CODE_DIR), "build_poi_poi_triples.py")
if os.path.exists(_extra):
    srcs.append(_extra)                 # only when the scripts sit in a group/ subdir
for src in srcs:
    dst = os.path.join(RUN, os.path.basename(src))
    shutil.copyfile(src, dst)
    os.chmod(dst, 0o644)
sys.path.insert(0, RUN)

GROUPS_DIR, KG_DIR = f"{WORK}/groups", f"{WORK}/kg"
json.dump(dict(RUN=RUN, DATA_DIR=DATA_DIR, CODE_DIR=CODE_DIR, WORK=WORK,
               GROUPS_DIR=GROUPS_DIR, KG_DIR=KG_DIR, DATASET=DATASET),
          open(f"{WORK}/_paths.json", "w"), indent=1)

print("scripts :", sorted(os.path.basename(p) for p in glob.glob(f"{RUN}/*.py")))
missing = {"build_groups.py", "affinity.py", "build_kg.py", "train_roth.py",
           "build_poi_poi_triples.py"} - {os.path.basename(p) for p in glob.glob(f"{RUN}/*.py")}
assert not missing, f"missing required scripts: {sorted(missing)}"
print("paths written to", f"{WORK}/_paths.json")

## 1 · Self-checks

Every script ships a synthetic fixture. Seconds each, and they fail loudly on a broken
environment instead of after a 30-minute run. Expect 11 + 12 + 4 + 9 = **36 PASS, 0 FAIL**.

In [ ]:
# --- self-sufficient preamble: survives a kernel restart or out-of-order execution ---
import os, sys, glob, json, shutil, subprocess
try:
    RUN, DATA_DIR, KG_DIR, GROUPS_DIR, WORK, DATASET   # noqa: F821
except NameError:
    _w = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./work"
    _pf = f"{_w}/_paths.json"
    assert os.path.exists(_pf), "run cell 0 (Locate inputs) first"
    globals().update(json.load(open(_pf)))
    sys.path.insert(0, RUN)

def run_stream(script, *args, check=True):
    """Run a pipeline script and stream its output into the cell AS IT HAPPENS.

    `subprocess.run` looks fine here and is a trap for anything slow: Python block-buffers
    stdout when it is a pipe rather than a terminal, so a 30-minute training run prints
    absolutely nothing until it exits and looks indistinguishable from a hang. `-u` plus
    line-by-line relaying fixes it.
    """
    cmd = [sys.executable, "-u", f"{RUN}/{script}", *map(str, args)]
    p = subprocess.Popen(cmd, cwd=RUN, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    out = []
    for line in p.stdout:
        print(line, end="", flush=True)
        out.append(line)
    p.wait()
    if check:
        assert p.returncode == 0, f"{script} failed (exit {p.returncode})"
    return p.returncode, "".join(out)

total_pass = total_fail = 0
for script in ("build_groups.py", "build_kg.py", "train_roth.py", "build_poi_poi_triples.py"):
    r = subprocess.run([sys.executable, f"{RUN}/{script}", "--self-check"],
                       capture_output=True, text=True, cwd=RUN)
    lines = [l for l in r.stdout.splitlines() if l.strip().startswith(("PASS", "FAIL"))]
    n_p = sum(l.strip().startswith("PASS") for l in lines)
    n_f = sum(l.strip().startswith("FAIL") for l in lines)
    total_pass += n_p; total_fail += n_f
    print(f"{script:<26} exit={r.returncode}  PASS={n_p}  FAIL={n_f}")
    for l in lines:
        if l.strip().startswith("FAIL"):
            print("   ", l.strip())
    if r.returncode != 0:
        print(r.stdout[-2000:]); print("STDERR:", r.stderr[-1000:])
print(f"\ntotal: {total_pass} PASS, {total_fail} FAIL")
assert total_fail == 0 and total_pass >= 30, "self-checks failed -- do not proceed"

## 2 · Stage 1 — group construction

Three regimes matching KCGRS's own taxonomy (Established / Occasional / Random), which that
paper names but never constructs. Filters, all chosen from measurements: venue rarity ≤50
visitors, ≥1 anchor–companion pair met ≥2×, causal category-intersection, every member ≥5
check-ins before *t*, joint history not 100% anchor.

Expect ~**35k train / 6.5k val / 13.4k test** group examples.

In [ ]:
# --- self-sufficient preamble: survives a kernel restart or out-of-order execution ---
import os, sys, glob, json, shutil, subprocess
try:
    RUN, DATA_DIR, KG_DIR, GROUPS_DIR, WORK, DATASET   # noqa: F821
except NameError:
    _w = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./work"
    _pf = f"{_w}/_paths.json"
    assert os.path.exists(_pf), "run cell 0 (Locate inputs) first"
    globals().update(json.load(open(_pf)))
    sys.path.insert(0, RUN)

def run_stream(script, *args, check=True):
    """Run a pipeline script and stream its output into the cell AS IT HAPPENS.

    `subprocess.run` looks fine here and is a trap for anything slow: Python block-buffers
    stdout when it is a pipe rather than a terminal, so a 30-minute training run prints
    absolutely nothing until it exits and looks indistinguishable from a hang. `-u` plus
    line-by-line relaying fixes it.
    """
    cmd = [sys.executable, "-u", f"{RUN}/{script}", *map(str, args)]
    p = subprocess.Popen(cmd, cwd=RUN, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    out = []
    for line in p.stdout:
        print(line, end="", flush=True)
        out.append(line)
    p.wait()
    if check:
        assert p.returncode == 0, f"{script} failed (exit {p.returncode})"
    return p.returncode, "".join(out)

run_stream("build_groups.py", "--data-dir", DATA_DIR, "--out-dir", GROUPS_DIR,
           "--dataset", DATASET)
for f in sorted(os.listdir(GROUPS_DIR)):
    print(f"  {os.path.getsize(f'{GROUPS_DIR}/{f}'):>12,}  {f}")


## 3 · Stage 2 — merged POI + group knowledge graph

The POI structural layer (taxonomy, spatial containment, proximity, sequence) is rebuilt from
`poi_metadata` + the TRAIN split; the group layer is added **on top of it**, not instead.

Sanity targets from your Stage-1 documentation: CATEGORY 500, SUBCATEGORY_OF 417,
HAS_CATEGORY 5,120, LOCALITY 142, REGION 2, IS_NEAR_TO ≈63k.

In [ ]:
# --- self-sufficient preamble: survives a kernel restart or out-of-order execution ---
import os, sys, glob, json, shutil, subprocess
try:
    RUN, DATA_DIR, KG_DIR, GROUPS_DIR, WORK, DATASET   # noqa: F821
except NameError:
    _w = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./work"
    _pf = f"{_w}/_paths.json"
    assert os.path.exists(_pf), "run cell 0 (Locate inputs) first"
    globals().update(json.load(open(_pf)))
    sys.path.insert(0, RUN)

def run_stream(script, *args, check=True):
    """Run a pipeline script and stream its output into the cell AS IT HAPPENS.

    `subprocess.run` looks fine here and is a trap for anything slow: Python block-buffers
    stdout when it is a pipe rather than a terminal, so a 30-minute training run prints
    absolutely nothing until it exits and looks indistinguishable from a hang. `-u` plus
    line-by-line relaying fixes it.
    """
    cmd = [sys.executable, "-u", f"{RUN}/{script}", *map(str, args)]
    p = subprocess.Popen(cmd, cwd=RUN, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    out = []
    for line in p.stdout:
        print(line, end="", flush=True)
        out.append(line)
    p.wait()
    if check:
        assert p.returncode == 0, f"{script} failed (exit {p.returncode})"
    return p.returncode, "".join(out)

run_stream("build_kg.py", "--data-dir", DATA_DIR, "--groups-dir", GROUPS_DIR,
           "--out-dir", KG_DIR, "--dataset", DATASET)

man = json.load(open(f"{KG_DIR}/kg_manifest.json"))
print("\nnode types:", man["node_types"])
for k, v in {"CATEGORY": 500, "LOCALITY": 142, "REGION": 2, "POI": 5120}.items():
    got = man["node_types"].get(k)
    print(f"  {k:<10} {got:>6}  (Stage-1 doc: {v})  {'OK' if got == v else 'DIFFERS'}")


## 4 · Stage 3 — RotH with the depth regulariser

`--depth-weight 0` reproduces the old behaviour (D1 ρ=+0.019, ABSENT). The defaults below are
what fixed it.

Watch `D1_rho` climb in the log. **A high ρ with non-monotonic per-depth radii is a false
pass** — that exact failure is what the self-check guards against, so check the radii table at
the end, not just ρ.

`EPOCHS=20` ≈ 5 min for a sanity pass (already reached ρ=+0.64); `120` ≈ 30 min (ρ=+0.85).

In [ ]:
# --- self-sufficient preamble: survives a kernel restart or out-of-order execution ---
import os, sys, glob, json, shutil, subprocess
try:
    RUN, DATA_DIR, KG_DIR, GROUPS_DIR, WORK, DATASET   # noqa: F821
except NameError:
    _w = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./work"
    _pf = f"{_w}/_paths.json"
    assert os.path.exists(_pf), "run cell 0 (Locate inputs) first"
    globals().update(json.load(open(_pf)))
    sys.path.insert(0, RUN)

def run_stream(script, *args, check=True):
    """Run a pipeline script and stream its output into the cell AS IT HAPPENS.

    `subprocess.run` looks fine here and is a trap for anything slow: Python block-buffers
    stdout when it is a pipe rather than a terminal, so a 30-minute training run prints
    absolutely nothing until it exits and looks indistinguishable from a hang. `-u` plus
    line-by-line relaying fixes it.
    """
    cmd = [sys.executable, "-u", f"{RUN}/{script}", *map(str, args)]
    p = subprocess.Popen(cmd, cwd=RUN, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    out = []
    for line in p.stdout:
        print(line, end="", flush=True)
        out.append(line)
    p.wait()
    if check:
        assert p.returncode == 0, f"{script} failed (exit {p.returncode})"
    return p.returncode, "".join(out)

EPOCHS = 120        # 20 for a quick pass (~5 min); 120 is ~30 min on 4 CPU cores

# Progress prints every 10 epochs. Watch D1_rho climb -- if it is still ABSENT past epoch 40,
# stop and raise --depth-weight rather than waiting for the run to end.
run_stream("train_roth.py", "--kg-dir", KG_DIR, "--data-dir", DATA_DIR, "--out-dir", KG_DIR,
           "--dataset", DATASET, "--epochs", EPOCHS, "--log-every", 10, "--max-eval", 4000,
           "--depth-weight", 5.0, "--depth-margin", 0.3, "--root-pull", 0.01)


## 5 · D1 — before vs after

The point of stage 3, measured on your data rather than taken on trust.

In [ ]:
# --- self-sufficient preamble: survives a kernel restart or out-of-order execution ---
import os, sys, glob, json, shutil, subprocess
try:
    RUN, DATA_DIR, KG_DIR, GROUPS_DIR, WORK, DATASET   # noqa: F821
except NameError:
    _w = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./work"
    _pf = f"{_w}/_paths.json"
    assert os.path.exists(_pf), "run cell 0 (Locate inputs) first"
    globals().update(json.load(open(_pf)))
    sys.path.insert(0, RUN)

def run_stream(script, *args, check=True):
    """Run a pipeline script and stream its output into the cell AS IT HAPPENS.

    `subprocess.run` looks fine here and is a trap for anything slow: Python block-buffers
    stdout when it is a pipe rather than a terminal, so a 30-minute training run prints
    absolutely nothing until it exits and looks indistinguishable from a hang. `-u` plus
    line-by-line relaying fixes it.
    """
    cmd = [sys.executable, "-u", f"{RUN}/{script}", *map(str, args)]
    p = subprocess.Popen(cmd, cwd=RUN, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    out = []
    for line in p.stdout:
        print(line, end="", flush=True)
        out.append(line)
    p.wait()
    if check:
        assert p.returncode == 0, f"{script} failed (exit {p.returncode})"
    return p.returncode, "".join(out)

import numpy as np, pandas as pd
from train_roth import d1_radial_hierarchy

meta = pd.read_csv(f"{DATA_DIR}/poi_metadata_{DATASET}.csv")
col = "category_path" if "category_path" in meta.columns else "category"
depths = np.array([len([x for x in str(s).split(">") if x.strip()])
                   for s in meta[col].fillna("")], dtype=float)

rows = []
old = glob.glob(f"{DATA_DIR}/**/poi_hyperbolic_embs.npy", recursive=True)
if old:
    rows.append(("OLD (shipped)", d1_radial_hierarchy(np.load(old[0]), depths)))
rows.append(("NEW (+depth reg)",
             d1_radial_hierarchy(np.load(f"{KG_DIR}/poi_hyperbolic_embs_{DATASET}.npy"), depths)))

for name, d in rows:
    print(f"{name:<18} rho={d['spearman']:+.4f}  {d['verdict']:<7} "
          f"norms [{d['norm_min']:.3f}, {d['norm_max']:.3f}]")
    print("   radius by depth: " +
          "  ".join(f"d{k}={v['mean_radius']:.3f}" for k, v in d["by_depth"].items()))

new = rows[-1][1]
radii = [v["mean_radius"] for v in new["by_depth"].values()]
mono = all(radii[i] < radii[i + 1] for i in range(len(radii) - 1))
print(f"\nmonotonic radii: {mono}    verdict: {new['verdict']}")
if not mono or new["verdict"] == "ABSENT":
    print("WARNING: raise --depth-weight or --depth-margin and re-run stage 3")

## 6 · Stage 4 — POI-POI triples for the alignment loss

Feeds the KG-triple-preservation term in `stage6b_run2_server.ipynb` §6b. `--kg-dir` reads the
tensor-format KG from stage 2 directly — no crosswalk file, no gpickle.

`--derive taxonomy` matters: both native POI-POI relations (`IS_NEAR_TO`, `FOLLOWED_BY`) are
**flat**, so without it the TransE term has no hierarchical signal to preserve. Expect ~137k
triples over 6 relations.

In [ ]:
# --- self-sufficient preamble: survives a kernel restart or out-of-order execution ---
import os, sys, glob, json, shutil, subprocess
try:
    RUN, DATA_DIR, KG_DIR, GROUPS_DIR, WORK, DATASET   # noqa: F821
except NameError:
    _w = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./work"
    _pf = f"{_w}/_paths.json"
    assert os.path.exists(_pf), "run cell 0 (Locate inputs) first"
    globals().update(json.load(open(_pf)))
    sys.path.insert(0, RUN)

def run_stream(script, *args, check=True):
    """Run a pipeline script and stream its output into the cell AS IT HAPPENS.

    `subprocess.run` looks fine here and is a trap for anything slow: Python block-buffers
    stdout when it is a pipe rather than a terminal, so a 30-minute training run prints
    absolutely nothing until it exits and looks indistinguishable from a hang. `-u` plus
    line-by-line relaying fixes it.
    """
    cmd = [sys.executable, "-u", f"{RUN}/{script}", *map(str, args)]
    p = subprocess.Popen(cmd, cwd=RUN, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    out = []
    for line in p.stdout:
        print(line, end="", flush=True)
        out.append(line)
    p.wait()
    if check:
        assert p.returncode == 0, f"{script} failed (exit {p.returncode})"
    return p.returncode, "".join(out)

run_stream("build_poi_poi_triples.py", "--kg-dir", KG_DIR,
           "--meta", f"{DATA_DIR}/poi_metadata_{DATASET}.csv",
           "--out-dir", KG_DIR, "--dataset", DATASET,
           "--derive", "taxonomy", "--max-per-relation", 40000)

v = json.load(open(f"{KG_DIR}/poi_relation_vocab_{DATASET}.json"))
print("\nrelations in the alignment triple set:")
for k, n in v["counts"].items():
    print(f"   {k:<22} {n:>8,}")


## 7 · Outputs

Save a **Version** so the next notebook can attach this one's output as a dataset.

What the LLaDA fine-tune consumes:

| File | Used as |
|---|---|
| `kg/poi_hyperbolic_embs_NYC.npy` | `EMB_FILE` in the notebook config |
| `kg/poi_poi_triples_NYC.pt` + `kg/poi_relation_vocab_NYC.json` | §6b `ALIGN_TRIPLES_FILE` / `ALIGN_RELVOCAB_FILE` |
| `groups/group_examples_{train,val,test}.jsonl` | the group task data |
| `groups/constructed_groups.csv` | group roster, for inspection and the paper's tables |

In [ ]:
# --- self-sufficient preamble: survives a kernel restart or out-of-order execution ---
import os, sys, glob, json, shutil, subprocess
try:
    RUN, DATA_DIR, KG_DIR, GROUPS_DIR, WORK, DATASET   # noqa: F821
except NameError:
    _w = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./work"
    _pf = f"{_w}/_paths.json"
    assert os.path.exists(_pf), "run cell 0 (Locate inputs) first"
    globals().update(json.load(open(_pf)))
    sys.path.insert(0, RUN)

def run_stream(script, *args, check=True):
    """Run a pipeline script and stream its output into the cell AS IT HAPPENS.

    `subprocess.run` looks fine here and is a trap for anything slow: Python block-buffers
    stdout when it is a pipe rather than a terminal, so a 30-minute training run prints
    absolutely nothing until it exits and looks indistinguishable from a hang. `-u` plus
    line-by-line relaying fixes it.
    """
    cmd = [sys.executable, "-u", f"{RUN}/{script}", *map(str, args)]
    p = subprocess.Popen(cmd, cwd=RUN, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    out = []
    for line in p.stdout:
        print(line, end="", flush=True)
        out.append(line)
    p.wait()
    if check:
        assert p.returncode == 0, f"{script} failed (exit {p.returncode})"
    return p.returncode, "".join(out)

for d in (GROUPS_DIR, KG_DIR):
    print(d)
    for f in sorted(os.listdir(d)):
        print(f"   {os.path.getsize(os.path.join(d, f)):>12,}  {f}")
    print()